# Forge - Notebook 03

# Document Chunking

This notebook splits processed documents into semantically meaningful chunks that are optimized for retrieval.

Output:
- Chunked JSON files
- Chunking statistics
- Chunking report

In [2]:
!pip install -q langchain-text-splitters

In [3]:
from pathlib import Path
import json
from uuid import uuid4

import pandas as pd

from google.colab import drive

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [4]:
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/forge")

CONFIG = {
    "project_root": PROJECT_ROOT,
    "processed": PROJECT_ROOT / "knowledge_base" / "processed",
    "chunks": PROJECT_ROOT / "knowledge_base" / "chunks",
}

In [6]:
for name, path in CONFIG.items():
    print(f"{name:15} : {'✓' if path.exists() else '✗'}")

project_root    : ✓
processed       : ✓
chunks          : ✓


In [7]:
processed_files = sorted(CONFIG["processed"].rglob("*.json"))

print(f"Processed Documents: {len(processed_files)}")

Processed Documents: 283


In [8]:
documents = []

for file_path in processed_files:
    with open(file_path, "r", encoding="utf-8") as file:
        documents.append(json.load(file))

print(f"Loaded {len(documents)} documents.")

Loaded 283 documents.


In [9]:
sample = documents[0]

print(json.dumps(sample, indent=2)[:2500])

{
  "technology": "anthropic_claude",
  "source": "api_reference",
  "path": "anthropic_claude/api_reference.txt",
  "content": "The Claude API is a RESTful API at https://api.anthropic.com that provides programmatic access to Claude models and Claude Managed Agents.\nNew to Claude? For direct model access, start with Get started and Working with Messages. For managed agent infrastructure, see the Claude Managed Agents quickstart.\nTo use the Claude API, you'll need:\nFor step-by-step setup instructions, see Get started.\nThe Claude API includes the following APIs:\nGeneral Availability:\nPOST /v1/messages)POST /v1/messages/batches)POST /v1/messages/count_tokens)GET /v1/models)Beta:\nPOST /v1/files, GET /v1/files)POST /v1/skills, GET /v1/skills)POST /v1/agents, GET /v1/agents)POST /v1/sessions, GET /v1/sessions/{id}/stream)POST /v1/environments, GET /v1/environments)For the complete API reference with all endpoints, parameters, and response schemas, explore the API reference pages list

In [10]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

print(f"Chunk Size    : {CHUNK_SIZE}")
print(f"Chunk Overlap : {CHUNK_OVERLAP}")

Chunk Size    : 1000
Chunk Overlap : 200


In [18]:
def chunk_document(document):
    chunks = text_splitter.split_text(document["content"])

    chunked_documents = []

    for index, chunk in enumerate(chunks):
        if len(chunk.strip()) < 100:
            continue

        chunked_documents.append({
            "chunk_id": str(uuid4()),
            "technology": document["technology"],
            "source": document["source"],
            "path": document["path"],
            "chunk_index": index,
            "text": chunk,
            "character_count": len(chunk)
        })

    return chunked_documents

In [19]:
sample_chunks = chunk_document(documents[0])

print(f"Number of Chunks: {len(sample_chunks)}")
print()

print(json.dumps(sample_chunks[0], indent=2)[:2000])

Number of Chunks: 9

{
  "chunk_id": "a5aeb7f3-b0c7-4f52-bc69-0ad911bcfaed",
  "technology": "anthropic_claude",
  "source": "api_reference",
  "path": "anthropic_claude/api_reference.txt",
  "chunk_index": 0,
  "text": "The Claude API is a RESTful API at https://api.anthropic.com that provides programmatic access to Claude models and Claude Managed Agents.\nNew to Claude? For direct model access, start with Get started and Working with Messages. For managed agent infrastructure, see the Claude Managed Agents quickstart.\nTo use the Claude API, you'll need:\nFor step-by-step setup instructions, see Get started.\nThe Claude API includes the following APIs:\nGeneral Availability:\nPOST /v1/messages)POST /v1/messages/batches)POST /v1/messages/count_tokens)GET /v1/models)Beta:\nPOST /v1/files, GET /v1/files)POST /v1/skills, GET /v1/skills)POST /v1/agents, GET /v1/agents)POST /v1/sessions, GET /v1/sessions/{id}/stream)POST /v1/environments, GET /v1/environments)For the complete API referenc

In [20]:
all_chunks = []

for document in documents:
    chunks = chunk_document(document)
    all_chunks.extend(chunks)

print(f"Documents : {len(documents)}")
print(f"Chunks    : {len(all_chunks)}")

Documents : 283
Chunks    : 3944


In [21]:
CONFIG["chunks"].mkdir(parents=True, exist_ok=True)

for chunk in all_chunks:
    technology_folder = CONFIG["chunks"] / chunk["technology"]
    technology_folder.mkdir(parents=True, exist_ok=True)

    output_file = technology_folder / f"{chunk['chunk_id']}.json"

    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(chunk, file, indent=2, ensure_ascii=False)

print(f"Saved {len(all_chunks)} chunks.")

Saved 3944 chunks.


In [22]:
report = pd.DataFrame([
    {
        "chunk_id": chunk["chunk_id"],
        "technology": chunk["technology"],
        "source": chunk["source"],
        "chunk_index": chunk["chunk_index"],
        "characters": chunk["character_count"],
    }
    for chunk in all_chunks
])

report_path = CONFIG["project_root"] / "knowledge_base" / "chunking_report.csv"

report.to_csv(report_path, index=False)

report.head()

,chunk_id,technology,source,chunk_index,characters
0,589088e1-c0cc-4a06-8fcf-dab5b6bd0cbe,anthropic_claude,api_reference,0,928
1,9d28e2c5-c636-4935-b850-4d0e2b06a673,anthropic_claude,api_reference,1,915
2,c97415bd-b252-46b3-85a2-40112f11001e,anthropic_claude,api_reference,2,966
3,b554ea3a-15a5-4462-ae60-cf037c47e8dd,anthropic_claude,api_reference,3,967
4,1ca07b62-2495-4505-92f9-3bd70db8919c,anthropic_claude,api_reference,4,898


In [23]:
print("=" * 50)
print("DOCUMENT CHUNKING SUMMARY")
print("=" * 50)

print(f"Processed Documents : {len(documents)}")
print(f"Total Chunks        : {len(all_chunks)}")
print(f"Chunk Size          : {CHUNK_SIZE}")
print(f"Chunk Overlap       : {CHUNK_OVERLAP}")

average_size = report["characters"].mean()
min_size = report["characters"].min()
max_size = report["characters"].max()

print(f"Average Chunk Size  : {average_size:.0f} characters")
print(f"Smallest Chunk      : {min_size} characters")
print(f"Largest Chunk       : {max_size} characters")

print(f"\nChunk Directory     : {CONFIG['chunks']}")
print(f"Chunk Report        : {report_path}")

DOCUMENT CHUNKING SUMMARY
Processed Documents : 283
Total Chunks        : 3944
Chunk Size          : 1000
Chunk Overlap       : 200
Average Chunk Size  : 843 characters
Smallest Chunk      : 100 characters
Largest Chunk       : 1000 characters

Chunk Directory     : /content/drive/MyDrive/forge/knowledge_base/chunks
Chunk Report        : /content/drive/MyDrive/forge/knowledge_base/chunking_report.csv


# Conclusion

This notebook successfully transformed the processed documentation corpus into retrieval-ready chunks.

Outputs:
- Chunked JSON files
- Chunking report
- Chunk metadata for vector indexing

These chunks will be embedded and indexed in the next notebook.